<a href="https://colab.research.google.com/github/samyak1729/genai-course/blob/main/SQL_Agent_Tutorial_claude.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a SQL Agent with LangChain & LangGraph

This notebook is a tutorial version of the guide available on [LangChain's official docs](https://docs.langchain.com/oss/python/langchain/sql-agent).

It also includes additional notes so that anyone reading the notebook can understand the full context without needing to refer back to the original tutorial.

---

## What are we building?

A **SQL Agent** is an LLM-powered system that can:
- Understand natural language questions about your data
- Automatically figure out which tables and columns to query
- Write and execute SQL queries
- Return a plain-English answer

We'll use the **Chinook** sample database (a digital music store) as our data source, and wire everything together using **LangChain**, **LangGraph**, and **OpenRouter**.

---

## Architecture Overview

```
User Question
     │
     ▼
  LLM (via OpenRouter)
     │  decides which tool to call
     ▼
SQL Tools (list tables → get schema → check query → run query)
     │
     ▼
  SQLite Database (Chinook.db)
     │
     ▼
  Final Answer
```

The agent loops through tool calls autonomously until it has enough information to answer the question.

## 1. Installation

We need three core packages:
- **`langchain`** — the main framework for chaining LLM calls and tools
- **`langgraph`** — builds the agent loop (the "react" loop: reason → act → observe → repeat)
- **`langchain-community`** — community integrations, including the SQL database utilities
- **`langchain-openrouter`** — LangChain wrapper for the OpenRouter API

> ⚠️ **Note:** After running this cell, you may need to **restart the runtime** (Runtime → Restart session) before the imports in later cells work correctly. This is because Python caches module locations at startup.

In [ ]:
!pip install -q langchain langgraph langchain-community
!pip install -q -U langchain-openrouter

## 2. Setting Up API Keys

We need two API keys:
1. **OpenRouter API key** — to access the LLM. Get one at [openrouter.ai](https://openrouter.ai)
2. **LangSmith API key** — for tracing and observability (optional but recommended). Get one at [smith.langchain.com](https://smith.langchain.com)

We use `getpass` here so your keys are never visible in the notebook output or stored in plain text in the cell. This is important if you ever share your notebook.

In [ ]:
import os
import getpass

# Prompt securely — input will be hidden
os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

# LangSmith for agent tracing (optional)
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API key (or press Enter to skip): ")

## 3. Initializing the LLM

We use **OpenRouter**, which is a unified API gateway that gives access to many LLMs (GPT-4, Claude, Mistral, Llama, etc.) through a single endpoint.

`ChatOpenRouter` is a LangChain-compatible wrapper, so it works seamlessly with all LangChain tools and agents.

The model `arcee-ai/trinity-large-preview:free` is a free model available on OpenRouter — great for tutorials. You can swap this out for any other model ID supported by OpenRouter.

In [ ]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(model="arcee-ai/trinity-large-preview:free")

print("Model initialized:", model.model_name)

## 4. Downloading the Sample Database

We'll use the **Chinook database**, a well-known sample SQLite database that represents a digital music store. It has tables for artists, albums, tracks, customers, invoices, and more.

We use two standard library modules:
- **`requests`** — for making HTTP requests to download the file
- **`pathlib`** — for clean, cross-platform file path handling

The code checks if the file already exists before downloading, so re-running this cell won't re-download unnecessarily.

In [ ]:
import requests
import pathlib

url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"
local_path = pathlib.Path("Chinook.db")

if local_path.exists():
    print(f"{local_path} already exists, skipping download.")
else:
    response = requests.get(url)
    if response.status_code == 200:
        local_path.write_bytes(response.content)
        print(f"File downloaded and saved as {local_path}")
    else:
        print(f"Failed to download the file. Status code: {response.status_code}")

## 5. Connecting to the Database with SQLDatabase

`SQLDatabase` from `langchain_community` is a thin wrapper around **SQLAlchemy**. It handles:
- Connecting to the database via a URI
- Inspecting available tables
- Fetching table schemas (column names, types, foreign keys)
- Running queries safely

Using this wrapper saves us from writing boilerplate SQLAlchemy code and makes the database compatible with LangChain's SQL tools out of the box.

The URI format for SQLite is: `sqlite:///path/to/file.db`

In [ ]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

print(f"Dialect: {db.dialect}")
print(f"Available tables: {db.get_usable_table_names()}")
print(f"\nSample query output:")
print(db.run("SELECT * FROM Artist LIMIT 5;"))

## 6. Creating the SQL Toolkit

The `SQLDatabaseToolkit` generates a set of **pre-built tools** that the agent can call. These tools are the agent's "hands" — they let it interact with the database step by step.

The toolkit provides 4 tools:

| Tool | What it does |
|---|---|
| `sql_db_list_tables` | Lists all tables in the database |
| `sql_db_schema` | Returns the schema + sample rows for given tables |
| `sql_db_query_checker` | Asks the LLM to double-check a query for errors before running |
| `sql_db_query` | Executes a SQL query and returns results |

The agent almost always uses them in this order: **list → schema → check → execute**. This mirrors how a human data analyst would approach an unfamiliar database.

In [ ]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=model)
tools = toolkit.get_tools()

print(f"Number of tools available: {len(tools)}\n")
for tool in tools:
    print(f" {tool.name}")
    print(f"   {tool.description}\n")

## 7. Writing the System Prompt

The **system prompt** is the instruction set we give the agent before it starts working. It defines:
- The agent's role and goal
- Safety rules (e.g., no destructive SQL like `DELETE` or `DROP`)
- Behavioral guidelines (e.g., always check tables first, always validate queries)

We use Python's `.format()` to inject two dynamic values:
- `{dialect}` — the SQL dialect (e.g., `sqlite`), so the agent generates compatible syntax
- `{top_k}` — the maximum number of rows to return by default, to keep responses concise

A well-written system prompt is critical. It's what keeps the agent focused and prevents it from doing things like querying all columns or making unsafe database changes.

In [ ]:
system_prompt = """
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run,
then look at the results of the query and return the answer. Unless the user
specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results.

You can order the results by a relevant column to return the most interesting
examples in the database. Never query for all the columns from a specific table,
only ask for the relevant columns given the question.

You MUST double check your query before executing it. If you get an error while
executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the
database.

To start you should ALWAYS look at the tables in the database to see what you
can query. Do NOT skip this step.

Then you should query the schema of the most relevant tables.
""".format(
    dialect=db.dialect,
    top_k=5,
)

print("System prompt ready.")
print(system_prompt)

## 8. Creating the Agent

We use `create_react_agent` from `langgraph.prebuilt` to build the agent.

**ReAct** stands for **Reason + Act**. It's a prompting pattern where the agent:
1. **Reasons** about what to do next
2. **Acts** by calling a tool
3. **Observes** the tool's output
4. Repeats until it has a final answer

LangGraph manages this loop as a stateful graph, which makes it easy to stream intermediate steps and inspect what the agent is doing at each stage.

`create_react_agent` takes:
- **`model`** — the LLM that does the reasoning
- **`tools`** — the list of tools it can call
- **`prompt`** — the system prompt we wrote above

In [ ]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    model,
    tools,
    prompt=system_prompt,
)

print("Agent created successfully.")

## 9. Running the Agent

Now let's ask the agent a natural language question about the database.

We use `agent.stream()` with `stream_mode="values"` to see **every step** the agent takes — not just the final answer. This is useful for learning and debugging.

Watch how the agent:
1. Lists available tables
2. Inspects the schema of relevant tables
3. Drafts and validates a SQL query
4. Executes the query
5. Interprets the result into a plain-English answer

In [ ]:
question = "Which genre on average has the longest tracks?"

print(f"Question: {question}\n")
print("=" * 60)

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

## 10. Try Your Own Questions

Now that the agent is set up, you can ask it anything about the Chinook database. Here are a few ideas to try:

```python
# Which artist has the most albums?
# What are the top 5 best-selling tracks?
# How many customers are from Brazil?
# What is the total revenue by country?
```

Just replace the `question` variable in the cell below and run it.

In [ ]:
question = "Which artist has the most albums in the database?"

print(f"Question: {question}\n")
print("=" * 60)

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

---

## Summary

Here's what we built and learned in this notebook:

| Step | What we did |
|---|---|
| Install | Set up LangChain, LangGraph, and OpenRouter |
| API Keys | Stored keys securely using `getpass` |
| LLM | Initialized a chat model via OpenRouter |
| Database | Downloaded the Chinook SQLite DB and connected via `SQLDatabase` |
| Tools | Generated 4 SQL tools using `SQLDatabaseToolkit` |
| System Prompt | Defined the agent's behavior and safety guardrails |
| Agent | Built a ReAct agent using `create_react_agent` from LangGraph |
| Query | Ran the agent and streamed its step-by-step reasoning |

### Key Takeaways

- The **ReAct loop** (Reason → Act → Observe) is what allows the agent to break a complex question into smaller steps autonomously.
- **`SQLDatabaseToolkit`** gives the agent the structured tools it needs to safely explore and query any SQL database.
- A **well-written system prompt** is essential — it defines the agent's role, constraints, and strategy.
- **LangSmith** tracing lets you observe every tool call and LLM response, which is invaluable for debugging and improving your agent.

### Next Steps

- Add **memory** so the agent can handle multi-turn conversations about the data
- Add **human-in-the-loop** confirmation before running any write queries
- Connect to a **real database** (PostgreSQL, MySQL, etc.) by changing the URI
- Explore the [LangGraph documentation](https://langchain-ai.github.io/langgraph/) for more advanced agent patterns